In [1]:
import requests
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os
import re

# Configuración de endpoints de la API de Amadeus (entorno de test)
AUTH_URL = "https://test.api.amadeus.com/v1/security/oauth2/token"
SEARCH_URL = "https://test.api.amadeus.com/v2/shopping/flight-offers"

# Credenciales (TP Sofi) - recordá regenerarlas y/o moverlas a variables de entorno
AMADEUS_API_KEY = "D8etJigXClzMlKAp3Duam26pdL7t8aBT"
AMADEUS_API_SECRET = "PO81V1rf5d8T21RF"

def parse_duration(duration_str):
    """Parsea una duración a minutos totales."""
    if not duration_str or not duration_str.startswith('PT'):
        return 0

    hours = 0
    minutes = 0

    time_part = duration_str[2:]

    h_match = re.search(r'(\d+)H', time_part)
    if h_match:
        hours = int(h_match.group(1))

    m_match = re.search(r'(\d+)M', time_part)
    if m_match:
        minutes = int(m_match.group(1))

    return hours * 60 + minutes
    
def get_access_token():
    """Obtiene el token de acceso para autenticar las llamadas a la API."""
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    body = {
        "grant_type": "client_credentials",
        "client_id": AMADEUS_API_KEY,
        "client_secret": AMADEUS_API_SECRET,
    }
    try:
        response = requests.post(AUTH_URL, headers=headers, data=body, timeout=10)
        response.raise_for_status()
        print("Token de acceso obtenido con éxito.")
        return response.json()["access_token"]
    except requests.exceptions.HTTPError as err:
        print(f"Error al obtener el token: {err.response.status_code} - {err.response.text}")
        return None
    except requests.exceptions.RequestException as err:
        print(f"Error de conexión al obtener el token: {err}")
        return None

#para resolver el error del vencimiento del token
def safe_request(url, params, token):
    """
    Hace un GET con el token actual.
    Si el token está vencido (401 con 'expired'), pide uno nuevo y reintenta.
    Devuelve: (response, token_actualizado)
    """
    headers = {"Authorization": f"Bearer {token}"}

    # Primer intento
    response = requests.get(url, headers=headers, params=params, timeout=20)

    # Si el token está vencido, pedimos uno nuevo y reintetamos una sola vez
    if response.status_code == 401 and "expired" in response.text:
        print(" Token vencido, generando uno nuevo...")
        token = get_access_token()
        headers = {"Authorization": f"Bearer {token}"}
        response = requests.get(url, headers=headers, params=params, timeout=20)

    return response, token


def collect_flight_data(
    access_token: str,
    routes: list,
    airlines: list,
    start_date: str,
    end_date: str,
    frequency: str,
):
    """
    Para cada ruta y aerolínea, consulta precios con Flight Offers Search.
    Maneja expiración de token con safe_request.
    """
    all_flights_data = []

    # Timestamp actual
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Fechas a consultar
    departure_dates = (
        pd.date_range(start=start_date, end=end_date, freq=frequency)
        .strftime("%Y-%m-%d")
        .tolist()
    )
    print(f"Se consultarán vuelos para las siguientes fechas de partida: {departure_dates}")

    for route in routes:
        print(f"\n--- Consultando ruta: {route['origin']} -> {route['destination']} ---")

        for date in departure_dates:

            for airline in airlines:

                search_params = {
                    "originLocationCode": route["origin"],
                    "destinationLocationCode": route["destination"],
                    "departureDate": date,
                    "adults": 1,
                    "max": 10,
                    "currencyCode": "USD",
                    "includedAirlineCodes": airline,
                }

                try:
                    # Usamos safe_request (y actualizamos access_token si venció)
                    search_response, access_token = safe_request(
                        SEARCH_URL,
                        search_params,
                        access_token,
                    )

                    search_response.raise_for_status()

                    flight_offers = search_response.json().get("data", [])

                    if not flight_offers:
                        print(
                            f"  - Aerolínea {airline}: No se encontraron vuelos "
                            f"para la partida el {date}."
                        )
                        continue

                    # Usamos la primera oferta de precio disponible
                    offer = flight_offers[0]

                    itineraries = offer.get("itineraries", [])
                    if not itineraries:
                        print(
                            f"  - Aerolínea {airline}: Oferta sin itinerarios "
                            f"para {route['origin']}-{route['destination']} el {date}."
                        )
                        continue

                    itinerary = itineraries[0]
                    segments = itinerary.get("segments", [])

                    # Layovers
                    layovers = []
                    if len(segments) > 1:
                        for seg in segments[:-1]:
                            arrival = seg.get("arrival", {})
                            layovers.append(arrival.get("iataCode"))

                    # Registro principal
                    flattened_data = {
                        "asset": f"{route['origin']}-{route['destination']}",
                        "downloadTime": current_time,
                        "departureDate": date,
                        "airline": airline,
                        "price": float(offer["price"]["grandTotal"]),
                        "stops": max(len(segments) - 1, 0),
                        "totalDuration": sum(parse_duration(segment.get('duration')) for segment in segments),
                        "layovers": layovers,
                    }

                    all_flights_data.append(flattened_data)

                    print(
                        f"  - Aerolínea {airline}: Obtenido precio para "
                        f"{route['origin']}-{route['destination']} el {date} "
                        f"por USD {flattened_data['price']}"
                    )

                except requests.exceptions.HTTPError as err:
                    print(
                        f"  - Error HTTP para {route['origin']}-{route['destination']}, "
                        f"{airline}, fecha {date}: "
                        f"{err.response.status_code} - {err.response.text}"
                    )

                except requests.exceptions.RequestException as e:
                    print(
                        f"  - Error de conexión para "
                        f"{route['origin']}-{route['destination']}, "
                        f"{airline}, fecha {date}: {e}"
                    )

    return all_flights_data


def store_data(data: list, path_output: str, timestamp: str):
    """Almacena los datos en un archivo Parquet, creando la carpeta de salida si no existe."""
    if not data:
        print("\nNo hay datos para guardar. Se omitió la creación del archivo Parquet.")
        return

    # Crear carpeta de salida si no existe
    os.makedirs(path_output, exist_ok=True)

    # Generar un DataFrame a partir de 'data'
    df = pd.DataFrame(data)

    # Generar una tabla a partir de 'df' usando pyarrow
    table = pa.Table.from_pandas(df)

    # Nombre de archivo incluyendo timestamp
    output_file = os.path.join(path_output, f"flight_data_{timestamp}.parquet")

    # Escribir el archivo Parquet
    pq.write_table(table, output_file)

    print(f"\nRecolección completa. Datos guardados exitosamente en: {output_file}")


def main():
    # Rutas a trackear
    routes_to_track = [
        {"origin": "JFK", "destination": "CDG"},
        {"origin": "BOS", "destination": "CDG"},
        {"origin": "MIA", "destination": "AMS"},
        {"origin": "ATL", "destination": "FCO"},
    ]

    # Aerolíneas a trackear (Delta, Air France, KLM)
    airlines_to_track = ["UA", "AF", "KL"]

    # Rango de fechas a consultar (podés ajustarlo cuando quieras)
    start_travel_date = "2025-11-23"
    end_travel_date = "2025-12-30"

    # Carpeta de salida (ruta en tu OneDrive)
    path_output = "."

    access_token = get_access_token()
    if access_token:
        collected_data = collect_flight_data(
            access_token,
            routes_to_track,
            airlines_to_track,
            start_travel_date,
            end_travel_date,
            frequency="D",
        )

        execution_timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
        store_data(collected_data, path_output, execution_timestamp)


if __name__ == "__main__":
    main()


Token de acceso obtenido con éxito.
Se consultarán vuelos para las siguientes fechas de partida: ['2025-11-23', '2025-11-24', '2025-11-25', '2025-11-26', '2025-11-27', '2025-11-28', '2025-11-29', '2025-11-30', '2025-12-01', '2025-12-02', '2025-12-03', '2025-12-04', '2025-12-05', '2025-12-06', '2025-12-07', '2025-12-08', '2025-12-09', '2025-12-10', '2025-12-11', '2025-12-12', '2025-12-13', '2025-12-14', '2025-12-15', '2025-12-16', '2025-12-17', '2025-12-18', '2025-12-19', '2025-12-20', '2025-12-21', '2025-12-22', '2025-12-23', '2025-12-24', '2025-12-25', '2025-12-26', '2025-12-27', '2025-12-28', '2025-12-29', '2025-12-30']

--- Consultando ruta: JFK -> CDG ---
  - Aerolínea UA: No se encontraron vuelos para la partida el 2025-11-23.
  - Aerolínea AF: Obtenido precio para JFK-CDG el 2025-11-23 por USD 362.0
  - Aerolínea KL: Obtenido precio para JFK-CDG el 2025-11-23 por USD 367.6
  - Aerolínea UA: Obtenido precio para JFK-CDG el 2025-11-24 por USD 334.0
  - Aerolínea AF: Obtenido precio

In [3]:
df = pd.read_parquet("./flight_data_2025_11_23_013411.parquet")

df.head()

,asset,downloadTime,departureDate,airline,price,stops,totalDuration,layovers
0,JFK-CDG,2025-11-23 00:52:45,2025-11-23,AF,362.0,0,435,[]
1,JFK-CDG,2025-11-23 00:52:45,2025-11-23,KL,367.6,1,664,[ATL]
2,JFK-CDG,2025-11-23 00:52:45,2025-11-24,UA,334.0,1,515,[YUL]
3,JFK-CDG,2025-11-23 00:52:45,2025-11-24,AF,362.0,0,425,[]
4,JFK-CDG,2025-11-23 00:52:45,2025-11-24,KL,362.0,1,510,[BOS]
